In [1]:
import pandas as pd
from general_information import read_balances

In [2]:
df_pay = pd.read_csv("data/03 Оплаты ХК.csv", sep=";")
print(df_pay.shape)
df_pay.head()

(1565019, 4)


,Номер,Дата оплаты,Сумма,Способ оплаты
0,15,24.01.2025,100,5.0
1,15,25.01.2026,1240,5.0
2,15,25.05.2025,"293,96",5.0
3,15,25.06.2025,10,5.0
4,15,26.02.2025,"289,56",5.0


In [3]:
balances = read_balances()
print(balances.shape)
cols_to_check = balances.columns.drop('ЛС')
balances = balances[(balances[cols_to_check] != 0).any(axis=1)]
print(balances.shape)

ids = balances['ЛС']
balances.head()

(150790, 46)
(136781, 46)


,ЛС,2025_1_start,2025_1_accr,2025_1_paid,2025_2_start,2025_2_accr,2025_2_paid,2025_3_start,2025_3_accr,2025_3_paid,...,2025_12_paid,2026_1_start,2026_1_accr,2026_1_paid,2026_2_start,2026_2_accr,2026_2_paid,2026_3_start,2026_3_accr,2026_3_paid
14,15.0,-130.94,520.50,100.00,289.56,572.55,289.56,572.55,-13.88,636.00,...,1800.00,1137.99,2738.34,1240.00,2636.33,1306.80,2636.33,1306.80,1603.80,1450.00
15,16.0,-833.24,308.83,300.00,-824.41,218.61,500.00,-1105.80,239.43,200.00,...,200.00,-363.78,483.12,200.00,-80.66,312.84,500.00,-267.82,423.72,200.00
16,17.0,249.84,260.25,249.84,260.25,579.49,260.25,579.49,485.80,0.00,...,1303.60,764.13,1445.40,764.13,1445.40,843.48,0.00,2288.88,443.52,1500.00
17,18.0,3.47,3.47,3.47,3.47,6.94,3.47,6.94,3.47,6.94,...,7.80,7.80,11.88,7.80,11.88,7.92,11.88,7.92,7.92,7.92
19,20.0,218.61,322.71,218.61,322.71,298.42,322.71,298.42,253.31,298.42,...,159.91,-31.20,0.00,51.96,-83.16,2471.04,0.00,2387.88,815.76,2475.00


In [4]:
df_pay = df_pay[df_pay['Номер'].isin(ids)]
print(df_pay.shape)

(1564671, 4)


In [5]:
from form_time_features import extract_payment_features

payment_features = extract_payment_features(df_pay, k=3, current_date=pd.to_datetime('2026-04-23'))
print(df_pay['Номер'].unique().shape, payment_features.shape)
print(f"Число неплатёжников: {(payment_features['Платежей_за_последние_3_мес'] == 0).sum()}")
payment_features.head()

(128691,) (128691, 3)
Число неплатёжников: 14204


,Номер,Дней_с_последнего_платежа,Платежей_за_последние_3_мес
0,15,26,3
1,16,36,2
2,17,42,1
3,18,39,2
4,20,40,1


In [6]:
from form_time_features import calculate_complex_features

complex_features = calculate_complex_features(df_pay, balances, k=3, curr_date=pd.to_datetime('2026-04-23'))
complex_features.head()

,Id,Days_Since_Clearance,Payment_Fraction_12M,Consecutive_Debt_Months,Payment_Accrual_Ratio_kM,Balance_Trend_Slope_3M,Debt_to_Avg_Accrual_3M,Days_Since_Advance_5th,Days_Since_Salary_20th
0,15,26.0,1.000000,0,1.010183,-20.595,-0.076050,18,3
1,16,36.0,1.000000,0,0.879441,47.980,-1.150679,18,3
2,17,99.0,0.818182,2,0.655342,394.440,0.866140,18,3
3,18,39.0,1.000000,0,1.000000,0.000,0.000000,18,3
4,20,40.0,0.818182,0,1.001603,-1.980,-0.079518,18,3


In [7]:
complex_features_cut = complex_features[complex_features['Days_Since_Clearance'] != 9999]
print(complex_features_cut.shape)
complex_features_cut.head()

(110547, 9)


,Id,Days_Since_Clearance,Payment_Fraction_12M,Consecutive_Debt_Months,Payment_Accrual_Ratio_kM,Balance_Trend_Slope_3M,Debt_to_Avg_Accrual_3M,Days_Since_Advance_5th,Days_Since_Salary_20th
0,15,26.0,1.000000,0,1.010183,-20.595,-0.076050,18,3
1,16,36.0,1.000000,0,0.879441,47.980,-1.150679,18,3
2,17,99.0,0.818182,2,0.655342,394.440,0.866140,18,3
3,18,39.0,1.000000,0,1.000000,0.000,0.000000,18,3
4,20,40.0,0.818182,0,1.001603,-1.980,-0.079518,18,3


In [8]:
# Проверка число сезонных признаков
from form_time_features import get_seasonality_features

current_date=pd.to_datetime('2026-04-23')
df_season = get_seasonality_features(current_date)
df_season.head()

,Is_Heating_Season,Season_Temperature_Cos,Season_Day_Cos,Season_Temperature_Sin,Season_Day_Sin
0,1,0.0,-0.3496,1.0,0.9369


In [9]:
import pandas as pd
import os

# путь к главному файлу
main_file = "data/14 Лимиты мер воздействия ХК.xlsx"

# читаем главный файл
limits_df = pd.read_excel(main_file)

# словарь для результатов
result = {}

for _, row in limits_df.iterrows():
    file_name = row.iloc[0]
    limit = row.iloc[1]

    # пропускаем пустые строки
    if pd.isna(file_name):
        continue

    file_path = os.path.join("data", file_name+".xlsx")

    # читаем файл без заголовков
    df_raw = pd.read_excel(file_path, header=None)

    # 1 строка — название операции
    operation_name = df_raw.iloc[0, 0]

    # 2 строка — заголовки
    df = df_raw.iloc[2:].copy()
    df.columns = df_raw.iloc[1]

    # чистка: убираем #Н/Д
    df = df[df["ЛС"].notna()]

    # сохраняем
    result[operation_name] = {
        "limit": limit,
        "data": df
    }

# теперь result — словарь с данными

In [10]:
result.keys()

dict_keys(['Автодозвон', 'E-mail', 'СМС', 'Обзвон оператором', 'Претензия', 'Выезд к абоненту', 'Уведомление о введении ограничения', 'Ограничение', 'Заявление о выдаче судебного приказа', 'Получение судебного приказа или ИЛ'])

In [11]:
result["Заявление о выдаче судебного приказа"]

{'limit': 400,
 'data': 1         ЛС                 Дата
 2         66  2025-07-30 00:00:00
 3         66  2025-07-30 00:00:00
 4         90  2025-11-24 00:00:00
 5         90  2025-11-24 00:00:00
 6        306  2025-09-25 00:00:00
 ...      ...                  ...
 8387  147017  2025-05-30 00:00:00
 8388  147017  2026-02-04 00:00:00
 8389  147017  2025-02-24 00:00:00
 8390  147017  2025-05-30 00:00:00
 8391  147017  2026-02-04 00:00:00
 
 [8390 rows x 2 columns]}

In [ ]:
df2 = pd.read_excel("data/01 Общая информация о ЛС ХК.xlsx", index_col=0)
df2.drop(columns=["Адрес (ГУИД)"], inplace=True)
df2